In [1]:
!find /kaggle/input -path "*phase-1-c-1*" \( -name "*.pt" -o -name "*.pkl" \) -exec ls -la {} \;

-rw-r--r-- 1 nobody nogroup 429235 Oct 15 23:11 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 429235 Oct 15 23:11 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/bpe_tokenizer.pkl
-rw-r--r-- 1 nobody nogroup 1331870311 Oct 15 23:12 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/ckpt.pt
-rw-r--r-- 1 nobody nogroup 443954142 Oct 15 23:12 /kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only/cs781_phase1_full/legal-gpt2-final/model_final.pt


In [2]:
# --- Phase 1 c-1 notebook, cell 1 (verbatim) ---
from collections import Counter
from tqdm.auto import tqdm
import pickle
import re
from collections import Counter, defaultdict

# --- Phase 1 c-1 notebook, cell 2: HF login ---
import os
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)

print("Hugging Face login successful.")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Hugging Face login successful.


In [3]:
# --- Phase 1 c-1 notebook, cell 8 (verbatim) ---
SPECIAL_TOKENS = ["<pad>", "<unk>", "<s>", "</s>"]  # fixed ids 0-3, never merged

class BPETokenizer:
    def __init__(self):
        self.id_to_bytes = {}
        self.bytes_to_id = {}
        self.merges = {}
        self.merge_order = []
        self.merge_ranks = {}  # Precomputed pair -> rank for fast encoding
        self.cache = {}        # Chunk lookup cache
        self._vocab_size = 0
        # Splits on whitespace+word or just whitespace.
        # Guarantees round-trip: "".join(chunks) == text
        self.split_pattern = re.compile(r'\s*\S+|\s+')

    def train(self, text: str, vocab_size: int):
        for i, tok in enumerate(SPECIAL_TOKENS):
            b = tok.encode("utf-8")
            self.id_to_bytes[i] = b
            self.bytes_to_id[b] = i
        next_id = len(SPECIAL_TOKENS)
        byte_to_base_id = {}
        for value in range(256):
            b = bytes([value])
            self.id_to_bytes[next_id] = b
            self.bytes_to_id[b] = next_id
            byte_to_base_id[value] = next_id
            next_id += 1
        chunks = self.split_pattern.findall(text)
        chunk_counts = Counter(chunks)
        chunk_to_ids = {
            chunk: tuple(byte_to_base_id[b] for b in chunk.encode("utf-8"))
            for chunk in chunk_counts
        }
        pair_counts = Counter()
        pair_to_chunks = defaultdict(set)
        def get_pairs(ids):
            return [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
        for chunk, ids in chunk_to_ids.items():
            count = chunk_counts[chunk]
            for pair in get_pairs(ids):
                pair_counts[pair] += count
                pair_to_chunks[pair].add(chunk)
        target = vocab_size
        num_merges = target - next_id
        pbar = tqdm(total=num_merges, desc="Training BPE")
        while next_id < target:
            if not pair_counts:
                break
            max_freq = max(pair_counts.values())
            if max_freq <= 1:
                break
            best_pair = min(p for p, c in pair_counts.items() if c == max_freq)
            new_id = next_id
            self.merges[best_pair] = new_id
            self.merge_order.append(best_pair)
            self.merge_ranks[best_pair] = len(self.merge_order) - 1
            self.id_to_bytes[new_id] = self.id_to_bytes[best_pair[0]] + self.id_to_bytes[best_pair[1]]
            self.bytes_to_id[self.id_to_bytes[new_id]] = new_id
            chunks_to_update = list(pair_to_chunks[best_pair])
            for chunk in chunks_to_update:
                old_ids = chunk_to_ids[chunk]
                count = chunk_counts[chunk]
                for pair in get_pairs(old_ids):
                    pair_counts[pair] -= count
                    if pair_counts[pair] <= 0:
                        del pair_counts[pair]
                    if chunk in pair_to_chunks[pair]:
                        pair_to_chunks[pair].remove(chunk)
                new_ids = []
                i = 0
                while i < len(old_ids):
                    if i < len(old_ids) - 1 and (old_ids[i], old_ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(old_ids[i])
                        i += 1
                new_ids = tuple(new_ids)
                chunk_to_ids[chunk] = new_ids
                for pair in get_pairs(new_ids):
                    pair_counts[pair] += count
                    pair_to_chunks[pair].add(chunk)
            next_id += 1
            pbar.update(1)
        pbar.close()
        self._vocab_size = next_id

    def encode(self, text: str):
        chunks = self.split_pattern.findall(text)
        out_ids = []
        for chunk in chunks:
            if chunk in self.cache:
                out_ids.extend(self.cache[chunk])
                continue
            chunk_bytes = chunk.encode("utf-8")
            ids = [self.bytes_to_id[bytes([b])] for b in chunk_bytes]
            while len(ids) > 1:
                pairs = [(ids[i], ids[i+1]) for i in range(len(ids) - 1)]
                best_pair = None
                best_rank = float('inf')
                for p in pairs:
                    rank = self.merge_ranks.get(p, float('inf'))
                    if rank < best_rank:
                        best_rank = rank
                        best_pair = p
                if best_pair is None:
                    break
                new_id = self.merges[best_pair]
                new_ids = []
                i = 0
                while i < len(ids):
                    if i < len(ids) - 1 and (ids[i], ids[i+1]) == best_pair:
                        new_ids.append(new_id)
                        i += 2
                    else:
                        new_ids.append(ids[i])
                        i += 1
                ids = new_ids
            self.cache[chunk] = ids
            out_ids.extend(ids)
        return out_ids

    def decode(self, ids):
        out = b"".join(self.id_to_bytes[i] for i in ids)
        return out.decode("utf-8", errors="replace")

    def get_vocab_size(self):
        return self._vocab_size

    @property
    def vocab_size(self):
        return self._vocab_size


def train_tokenizer(train_text: str, vocab_size: int):
    tok = BPETokenizer()
    tok.train(train_text, vocab_size)
    return tok

In [4]:
# --- Phase 1 c-1 notebook, cell 21 (verbatim) ---
import math
from dataclasses import dataclass, asdict
import torch
import torch.nn as nn
import torch.nn.functional as F


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, max_seq_len: int, dropout: float = 0.0, attention_scale=None):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        # None means the standard Transformer scaling 1/sqrt(d_head).
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)

        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)

        causal_mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal_mask, persistent=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """x: (B, T, d_model) -> (B, T, d_model)"""
        B, T, C = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        q = q.view(B, T, self.n_heads, self.d_head)
        q = q.transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.d_head)
        k = k.transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.d_head)
        v = v.transpose(1, 2)

        scores = q @ k.transpose(-2, -1) * self.attention_scale
        mask = self.causal_mask[:T, :T]
        scores = scores.masked_fill(~mask, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)
        out = attn @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.out_proj(out)
        out = self.resid_dropout(out)
        return out


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 1024
    n_layers: int = 6
    n_heads: int = 8
    n_kv_heads: int = 4          # == n_heads for plain MHA; unused in this version
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"
    attention_scale: float = None
    norm_first: bool = False     # True = pre-norm, False = post-norm


def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    """Return a (max_len, d_model) tensor of the classic Transformer sinusoidal PE."""
    pe = torch.zeros(max_len, d_model)
    for a in range(max_len):
        for b in range(d_model):
            i = b // 2
            angle = a / (10000 ** (2 * i / d_model))
            if b % 2 == 0:
                pe[a][b] = math.sin(angle)
            else:
                pe[a][b] = math.cos(angle)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout, attention_scale=cfg.attention_scale)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)
        self.norm_first = cfg.norm_first

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))      # FIX: your A3 code used self.ln1 here (ln2 was never used)
        else:
            x = self.ln1(x + self.attn(x))     # FIX: your A3 code used self.ln2 here (ln1 was never used)
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)

        if cfg.pos_emb_type == "sinusoidal":
            pe = sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model)
            self.register_buffer("pos_emb", pe, persistent=False)

        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying

    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """idx: (B, T) LongTensor of token ids -> logits (B, T, vocab_size)."""
        B, T = idx.shape
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + self.pos_emb[:T, :]
        for block in self.blocks:
            x = block(x)

        x = self.ln_f(x)
        logits = self.head(x)
        return logits

In [5]:
# =====================================================================================
# CELL A — load the Phase 1 c-1 decoder and its tokenizer
# Expect <All keys matched successfully>, 110.98M params, vocab 10000, d_model 1024, 8 layers, max_seq_len 2048
# =====================================================================================
import torch, pickle, json, re, os
ROOT = "/kaggle/input/notebooks/rudrajain2006/phase-1-c-1-decoder-only"   # TODO: the folder the find command showed (up to, not including, cs781_phase1_full)
device = torch.device("cuda")

# [c-1] checkpoint folder is legal-gpt2-final (c-6 was model_encdec); class is DecoderLM + LMConfig
tok = pickle.load(open(f"{ROOT}/cs781_phase1_full/bpe_tokenizer.pkl", "rb"))
ck = torch.load(f"{ROOT}/cs781_phase1_full/legal-gpt2-final/model_final.pt", map_location="cpu", weights_only=False)
cfg = LMConfig(**ck["config"])
lm = DecoderLM(cfg); print(lm.load_state_dict(ck["model"])); lm.to(device).eval()
del ck
print("params", sum(p.numel() for p in lm.parameters())/1e6, "M | vocab", tok.vocab_size,
      "| d_model", cfg.d_model, "| layers", cfg.n_layers, "| d_ff", cfg.d_ff, "| max_seq_len", cfg.max_seq_len,
      "| norm_first", cfg.norm_first, "| dropout", cfg.dropout)

<All keys matched successfully>
params 110.979072 M | vocab 10000 | d_model 1024 | layers 8 | d_ff 4096 | max_seq_len 2048 | norm_first False | dropout 0.1


In [6]:
# =====================================================================================
# CELL B — per-token vectors from the c-1 decoder + sanity check
# [c-1] c-1 has no encoder: the features are the decoder's last hidden states (after ln_f),
#       with the LM head skipped. Each token only sees the tokens to its LEFT (causal).
# =====================================================================================
def features(idx):                        # idx: (B, T) LongTensor, T <= 2048
    x = lm.tok_emb(idx)
    if lm.cfg.pos_emb_type == "sinusoidal":
        x = x + lm.pos_emb[:idx.size(1), :]
    for blk in lm.blocks:
        x = blk(x)
    return lm.ln_f(x)                     # (B, T, 1024)

# Sanity check: the LM head on these features must give the full model's logits
idx = torch.randint(4, cfg.vocab_size, (1, 64), device=device)
with torch.no_grad():
    f = features(idx); diff = (lm.head(f) - lm(idx)).abs().max().item()
print("max diff", diff, tuple(f.shape))   # expect 0.0, (1, 64, 1024)

max diff 0.0 (1, 64, 1024)


In [7]:
# =====================================================================================
# CELL C — Phase 3 (RR) data: load CL + IT train/dev/test and the label vocabulary
# =====================================================================================
import json
from collections import Counter
from huggingface_hub import hf_hub_download

def get3(name):
    p = hf_hub_download("Exploration-Lab/CS-781-Capstone", f"Phase_3_RR/{name}", repo_type="dataset")
    return json.load(open(p, encoding="utf-8"))

# Look at the real layout first (train and test differ)
_raw = get3("Train/CL_train.json")
print("train layout:", type(_raw).__name__, len(_raw))
_t = get3("Test/CL_test.json")
print("test layout:", type(_t).__name__, len(_t),
      str(_t[0] if isinstance(_t, list) else next(iter(_t.items())))[:300])

def as_docs(raw):
    """-> {doc_id: {"text": [sentences], "labels": [labels] or None}}.
    Accepts: a list of {"id","text",("labels")} dicts, {doc_id: {...}}, {doc_id: [sentences]},
    a one-key wrapper like {"data": [...]}, or columns {"id": [...], "text": [[...], ...]}."""
    while isinstance(raw, dict) and len(raw) == 1:
        v = next(iter(raw.values()))
        if (isinstance(v, list) and v and isinstance(v[0], dict)) or \
           (isinstance(v, dict) and v and all(isinstance(x, (dict, list)) for x in v.values())):
            raw = v
        else:
            break
    if isinstance(raw, dict) and "id" in raw and "text" in raw and isinstance(raw["id"], list):
        labs = raw.get("labels") or [None] * len(raw["id"])
        raw = [{"id": i, "text": t, "labels": l} for i, t, l in zip(raw["id"], raw["text"], labs)]
    if isinstance(raw, list):
        items = [(d.get("id", n), d) for n, d in enumerate(raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(f"unexpected file layout {type(raw).__name__}: print it and adapt as_docs")
    docs = {}
    for k, d in items:
        if isinstance(d, list):                       # {doc_id: [sentences]}
            d = {"text": d}
        assert isinstance(d, dict), f"doc {k}: unexpected entry {type(d).__name__}"
        k = str(d.get("id", k))
        assert k not in docs, f"duplicate doc id {k}"
        text = d.get("text", d.get("sentences"))
        assert isinstance(text, list), f"doc {k}: no list of sentences under 'text' (keys {list(d)})"
        labs = d.get("labels")
        docs[k] = {"text": list(text), "labels": list(labs) if labs is not None else None}
    return docs

DOMAINS = ("CL", "IT")
train = {dom: as_docs(get3(f"Train/{dom}_train.json")) for dom in DOMAINS}
dev   = {dom: as_docs(get3(f"Train/{dom}_dev.json"))   for dom in DOMAINS}
test  = {dom: as_docs(get3(f"Test/{dom}_test.json"))   for dom in DOMAINS}
print("first test ids:", list(test["CL"])[:3])

# Label vocabulary: a list, {"labels": [...]}, {label: id} or {id: label}
lv = get3("Train/label_vocab.json")
if isinstance(lv, dict) and len(lv) == 1 and isinstance(next(iter(lv.values())), list):
    lv = next(iter(lv.values()))
if isinstance(lv, dict):
    if all(isinstance(v, int) for v in lv.values()):
        RR_LABELS = sorted(lv, key=lv.get)                    # {label: id}
    else:
        RR_LABELS = [lv[k] for k in sorted(lv, key=int)]      # {id: label}
else:
    RR_LABELS = list(lv)
print(RR_LABELS)
assert len(RR_LABELS) == 13 and "None" in RR_LABELS, "check label_vocab.json by hand"
LAB_SET = set(RR_LABELS)

# Sanity checks: lengths match, labels are in the vocab, "None" is a string not null
for split_name, split in (("train", train), ("dev", dev), ("test", test)):
    for dom in DOMAINS:
        docs = split[dom]
        n_sent = sum(len(d["text"]) for d in docs.values())
        print(f"{split_name:5s} {dom}: {len(docs):4d} docs, {n_sent:6d} sentences")
        for k, d in docs.items():
            if d["labels"] is None:
                continue
            assert len(d["text"]) == len(d["labels"]), f"{split_name}/{dom}/{k}: text vs labels length"
            bad = [l for l in d["labels"] if l not in LAB_SET]
            assert not bad, f"{split_name}/{dom}/{k}: labels not in vocab {set(map(repr, bad))}"

# Label counts per domain
for split_name, split in (("train", train), ("dev", dev)):
    for dom in DOMAINS:
        c = Counter(l for d in split[dom].values() for l in d["labels"])
        print(split_name, dom, {l: c.get(l, 0) for l in RR_LABELS})

# Do doc IDs collide? (one prediction.json holds both domains)
for split_name, split in (("train", train), ("dev", dev), ("test", test)):
    print(f"{split_name}: CL∩IT ids = {len(set(split['CL']) & set(split['IT']))}")
for dom in DOMAINS:
    print(f"{dom}: train∩dev ids = {len(set(train[dom]) & set(dev[dom]))}")

CL_train.json:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

train layout: list 40


CL_test.json:   0%|          | 0.00/281k [00:00<?, ?B/s]

test layout: dict 5 ('HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', ['Manmohan, J. 1 .', 'Present writ petition has been filed, challenging the notice dated 26th February, 2016 and respondent Number 1s order dated 18th February, 2016 whereby the respondent-CCI has directed the opp


IT_train.json:   0%|          | 0.00/1.51M [00:00<?, ?B/s]

CL_dev.json:   0%|          | 0.00/336k [00:00<?, ?B/s]

IT_dev.json:   0%|          | 0.00/165k [00:00<?, ?B/s]

IT_test.json:   0%|          | 0.00/194k [00:00<?, ?B/s]

first test ids: ['HC_Raghupati_Singhania__vs__Competition_Commission_ofDE201616051616063761COM513489', 'COM_TPM_Consultants_Pvt_Ltd_vs_Competition_Commission_TA2016110716153210127COM903224', 'SC_Ultra Tech Cement Ltd vs Competition Commission of India and Ors 12062013 SC']


label_vocab.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']
train CL:   40 docs,  10593 sentences
train IT:   40 docs,   6252 sentences
dev   CL:    5 docs,   1396 sentences
dev   IT:    5 docs,    746 sentences
test  CL:    5 docs,   1339 sentences
test  IT:    5 docs,    858 sentences
train CL {'Fact': 2979, 'Issue': 17, 'ArgumentPetitioner': 976, 'ArgumentRespondent': 996, 'Statute': 367, 'Dissent': 284, 'PrecedentReliedUpon': 1305, 'PrecedentNotReliedUpon': 209, 'PrecedentOverruled': 0, 'RulingByLowerCourt': 438, 'RatioOfTheDecision': 2856, 'RulingByPresentCourt': 113, 'None': 53}
train IT {'Fact': 1119, 'Issue': 90, 'ArgumentPetitioner': 637, 'ArgumentRespondent': 352, 'Statute': 732, 'Dissent': 0, 'PrecedentReliedUpon': 640, 'PrecedentNotReliedUpon': 360, 'PrecedentOverruled': 3, 'RulingByLowerCourt': 436, 'Rati

In [8]:
# =====================================================================================
# CELL D — sentence tokenisation, local scorer, prediction writer, majority baseline
# =====================================================================================
import json, os, zipfile
import numpy as np
from collections import Counter
from tqdm.auto import tqdm

UNK_ID = SPECIAL_TOKENS.index("<unk>")   # 1
EOS_ID = SPECIAL_TOKENS.index("</s>")    # 3

PREFIX_SPACE = False   # TODO True/False: encode " " + sentence? (same as your c-6 run)
SEP_TOKEN    = "</s>"   # TODO "none" or "</s>": between sentences in document mode (same as your c-6 run)
assert PREFIX_SPACE is not None and SEP_TOKEN in ("none", "</s>"), "set PREFIX_SPACE and SEP_TOKEN"
SEP_ID = None if SEP_TOKEN == "none" else EOS_ID

def encode_sentence(s):
    ids = tok.encode((" " + s) if PREFIX_SPACE else s)
    return ids if ids else [UNK_ID]   # an empty sentence still needs one token to get a prediction

# Tokenise every sentence once: d["ids"] = [ids of sentence 0, ...] (re-run if PREFIX_SPACE changes)
for split in (train, dev, test):
    for dom in DOMAINS:
        for d in tqdm(split[dom].values(), desc="tokenising sentences", leave=False):
            d["ids"] = [encode_sentence(s) for s in d["text"]]

# Length statistics, to choose MAX_SENT_TOKENS / WINDOW (<= 2048 for c-1)   [c-1] limit 2048, not 1536
for dom in DOMAINS:
    sl = np.array([len(x) for d in train[dom].values() for x in d["ids"]])
    dl = np.array([sum(map(len, d["ids"])) for d in train[dom].values()])
    print(f"{dom} sentence tokens: median {np.median(sl):.0f} p95 {np.percentile(sl, 95):.0f} max {sl.max()} | "
          f"doc tokens: median {np.median(dl):.0f} p95 {np.percentile(dl, 95):.0f} max {dl.max()}")


# Local scorer: macro-F1 over the labels that appear in gold or predictions (sklearn convention)
def macro_scores(gold_lists, pred_lists):
    tp, fp, fn = Counter(), Counter(), Counter()
    for g, p in zip(gold_lists, pred_lists):
        assert len(g) == len(p), "one prediction per sentence"
        for a, b in zip(g, p):
            if a == b: tp[a] += 1
            else:      fp[b] += 1; fn[a] += 1
    used = [l for l in RR_LABELS if tp[l] + fp[l] + fn[l] > 0]
    per = {}
    for l in used:
        P = tp[l] / (tp[l] + fp[l]) if tp[l] + fp[l] else 0.0
        R = tp[l] / (tp[l] + fn[l]) if tp[l] + fn[l] else 0.0
        per[l] = (P, R, 2 * P * R / (P + R) if P + R else 0.0, tp[l] + fn[l])   # last = gold support
    m = lambda i: sum(v[i] for v in per.values()) / max(1, len(per))
    return {"p": m(0), "r": m(1), "f1": m(2), "per": per}

def evaluate(pred_by_domain, docs_by_domain, show=True):
    """pred_by_domain: {dom: {doc_id: [label, ...]}}. Scores each domain and ALL together."""
    res, allg, allp = {}, [], []
    for dom, docs in docs_by_domain.items():
        g = [d["labels"] for d in docs.values()]
        p = [pred_by_domain[dom][k] for k in docs]
        res[dom] = macro_scores(g, p); allg += g; allp += p
    res["ALL"] = macro_scores(allg, allp)
    if show:
        for k, v in res.items():
            print(f"{k:3s} macro-F1 {v['f1']:.4f}  P {v['p']:.4f}  R {v['r']:.4f}")
    return res

def per_label_table(res):
    for l in RR_LABELS:
        row = [f"{res[k]['per'][l][2]:.3f} (n={res[k]['per'][l][3]})" if l in res[k]["per"] else "  -  "
               for k in ("CL", "IT")]
        print(f"  {l:24s} CL F1 {row[0]:16s} IT F1 {row[1]}")

def confusion(pred_by_domain, docs_by_domain, dom):
    """Rows = gold, columns = predicted (counts), for the error analysis in the report."""
    idx = {l: i for i, l in enumerate(RR_LABELS)}
    M = np.zeros((len(RR_LABELS), len(RR_LABELS)), dtype=int)
    for k, d in docs_by_domain[dom].items():
        for g, p in zip(d["labels"], pred_by_domain[dom][k]):
            M[idx[g], idx[p]] += 1
    short = [l[:6] for l in RR_LABELS]
    print(f"{dom} confusion (rows gold, cols pred)\n" + " " * 8 + " ".join(f"{s:>6s}" for s in short))
    for s, row in zip(short, M):
        print(f"{s:>7s} " + " ".join(f"{v:6d}" for v in row))
    return M


# Writer: one prediction.json with BOTH domains, format checks, zipped for Codabench
def write_prediction(pred_by_domain, test_docs, out_dir="/kaggle/working"):
    out = {}
    for dom, docs in test_docs.items():
        for doc_id, d in docs.items():
            labs = pred_by_domain[dom][doc_id]
            assert len(labs) == len(d["text"]), f"{dom}/{doc_id}: wrong number of predictions"
            assert all(isinstance(l, str) and l in LAB_SET for l in labs), f"{dom}/{doc_id}: bad label"
            assert doc_id not in out, f"doc id {doc_id} is in both CL and IT test files: ask the TAs"
            out[doc_id] = list(labs)
    assert len(out) == sum(len(v) for v in test_docs.values()), "every test doc ID, no extras"
    path = os.path.join(out_dir, "prediction.json")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False)
    zpath = os.path.join(out_dir, "submission.zip")
    with zipfile.ZipFile(zpath, "w", compression=zipfile.ZIP_DEFLATED) as z:
        z.write(path, arcname="prediction.json")
    with zipfile.ZipFile(zpath) as z:
        assert z.namelist() == ["prediction.json"], z.namelist()
    print(f"{len(out)} docs, {sum(map(len, out.values()))} sentences -> {zpath}")


# Majority baseline: checks the scorer and writer end to end before any modelling
majority = {dom: Counter(l for d in train[dom].values() for l in d["labels"]).most_common(1)[0][0]
            for dom in DOMAINS}
print("majority label per domain:", majority)
base_dev = {dom: {k: [majority[dom]] * len(d["text"]) for k, d in dev[dom].items()} for dom in DOMAINS}
evaluate(base_dev, dev)
base_test = {dom: {k: [majority[dom]] * len(d["text"]) for k, d in test[dom].items()} for dom in DOMAINS}
write_prediction(base_test, test, out_dir="/kaggle/working")   # format check only; don't submit it

tokenising sentences:   0%|          | 0/40 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/40 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

tokenising sentences:   0%|          | 0/5 [00:00<?, ?it/s]

CL sentence tokens: median 37 p95 97 max 1116 | doc tokens: median 5796 p95 48127 max 83276
IT sentence tokens: median 36 p95 113 max 1363 | doc tokens: median 6079 p95 17740 max 22313
majority label per domain: {'CL': 'Fact', 'IT': 'RatioOfTheDecision'}
CL  macro-F1 0.0303  P 0.0178  R 0.1000
IT  macro-F1 0.0243  P 0.0140  R 0.0909
ALL macro-F1 0.0428  P 0.0302  R 0.0769
10 docs, 2197 sentences -> /kaggle/working/submission.zip


In [9]:
# =====================================================================================
# CELL E0a — shared plumbing for the c-1 Phase 3 (RR) experiments (part 1 of 2)
# Run E0a then E0b once after Cells 0, A, B, C, D. Then run the technique cell (E2 LORMA).
# Lines marked [c-1] are where this differs from the c-6 version.
# =====================================================================================
import math, time, random
from collections import defaultdict
import torch, torch.nn as nn, torch.nn.functional as F
from tqdm.auto import tqdm

N_LABELS = len(RR_LABELS)
LAB2ID = {l: i for i, l in enumerate(RR_LABELS)}
IGNORE = -100            # sentences with no gold label (test)
FEAT_DIM = cfg.d_model   # [c-1] 1024 (c-6 was 560)
PAD_ID = SPECIAL_TOKENS.index("<pad>")   # [c-1] 0; c-1 is causal, so right padding is safe (see E0b)
# [c-1] nothing to drop: c-1 is decoder-only. lm.head (tied to tok_emb) is simply never called.


# [c-1] decoder path (same as Cell B features(), on any backbone object)
def decoder_forward(backbone, idx):
    T = idx.size(1)
    assert T <= backbone.cfg.max_seq_len, "sequence longer than 2048"
    x = backbone.tok_emb(idx)
    if backbone.cfg.pos_emb_type == "sinusoidal":
        x = x + backbone.pos_emb[:T, :]
    for blk in backbone.blocks:                  # 8 causal decoder Blocks
        x = blk(x)
    return backbone.ln_f(x)                      # (B, T, 1024); LM head skipped


def decoder_linears(backbone, which):
    """Yield (parent, attr, nn.Linear) for the chosen linears in every decoder block.
    [c-1] which is a subset of {"q_proj", "k_proj", "v_proj", "out_proj", "ffn_in", "ffn_out"}.
    c-1 attention has SEPARATE q_proj/k_proj/v_proj (1024 -> 1024 each, no bias),
    not c-6's fused qkv_proj. FFN: ffn_in 1024 -> 4096, ffn_out 4096 -> 1024 (with bias)."""
    for blk in backbone.blocks:                  # [c-1] lm.blocks, block.attn (c-6: encoder_layers, self_attn)
        for w in which:
            if w in ("q_proj", "k_proj", "v_proj", "out_proj"):
                parent, attr = blk.attn, w
            elif w == "ffn_in":
                parent, attr = blk.ffn.net, "0"
            elif w == "ffn_out":
                parent, attr = blk.ffn.net, "2"
            else:
                raise ValueError(f"{w}: c-1 targets are q_proj, k_proj, v_proj, out_proj, ffn_in, ffn_out")
            yield parent, attr, getattr(parent, attr)


def wrap_linears(backbone, which, make_wrapper):
    """Replace each chosen linear by make_wrapper(linear). Returns how many were wrapped."""
    todo = list(decoder_linears(backbone, which))
    for parent, attr, lin in todo:
        assert isinstance(lin, nn.Linear), f"{attr} is already wrapped (re-run Cell A to start fresh)"
        setattr(parent, attr, make_wrapper(lin))
    return len(todo)


class RRModel(nn.Module):
    """c-1 decoder -> token vectors -> pool each sentence's tokens -> head -> one logit row per sentence."""
    def __init__(self, backbone, pool, head):
        super().__init__()
        self.backbone, self.pool, self.head = backbone, pool, head
    def forward(self, ids, spans):
        h = decoder_forward(self.backbone, ids)                         # [c-1] (B, T, 1024)
        vecs = [self.pool(h[b, s:e]) for b, sp in enumerate(spans) for (s, e) in sp]   # pads never pooled
        return self.head(torch.stack(vecs))                             # (S, 13)


# Examples. One example = one decoder input + the sentences it predicts.
#   mode="sentence": every sentence is its own input (first max_sent_tokens tokens).
#   mode="document": a document's sentences are joined (SEP_ID between them if set) and cut
#     into windows of `window` tokens every `stride` tokens; each sentence is predicted from ONE window.
#     [c-1] c-1 is causal, so only LEFT context reaches a sentence: the owner is the window that
#     holds the sentence whole with the most tokens before it (c-6 used the tighter of both sides).
#     If no window holds it whole, the largest overlap wins.
def make_windows(n, window, stride):
    if n <= window:
        return [(0, n)]
    starts = list(range(0, n - window, stride)) + [n - window]
    return [(s, s + window) for s in starts]


def build_examples(docs_by_domain, mode, *, max_sent_tokens=None, window=None, stride=None):
    assert mode in ("sentence", "document"), mode
    examples = []
    for dom, docs in docs_by_domain.items():
        for doc_id, d in tqdm(docs.items(), desc=f"{dom} examples ({mode})", leave=False):
            labs = [LAB2ID[l] for l in d["labels"]] if d["labels"] is not None else [IGNORE] * len(d["ids"])
            if mode == "sentence":
                for i, ids in enumerate(d["ids"]):
                    ids = ids[:max_sent_tokens]
                    examples.append({"ids": ids, "spans": [(0, len(ids))],
                                     "keys": [(dom, doc_id, i)], "labels": [labs[i]]})
                continue
            stream, sp = [], []
            for i, ids in enumerate(d["ids"]):
                if i > 0 and SEP_ID is not None:
                    stream.append(SEP_ID)
                sp.append((len(stream), len(stream) + len(ids)))
                stream += ids
            if not stream:
                continue
            wins = make_windows(len(stream), window, stride)
            owner = []
            for s, e in sp:
                def score(w):
                    ws, we = wins[w]
                    if ws <= s and e <= we:
                        return s - ws                                 # [c-1] whole inside: left context only
                    return min(e, we) - max(s, ws) - 10 ** 9          # partial: largest overlap wins
                owner.append(max(range(len(wins)), key=score))
            for w, (ws, we) in enumerate(wins):
                own = [i for i in range(len(sp)) if owner[i] == w]
                if not own:
                    continue
                examples.append({
                    "ids": stream[ws:we],
                    "spans": [(max(sp[i][0], ws) - ws, min(sp[i][1], we) - ws) for i in own],
                    "keys": [(dom, doc_id, i) for i in own],
                    "labels": [labs[i] for i in own]})
    return examples

In [10]:
# =====================================================================================
# CELL E0b — shared plumbing for the c-1 Phase 3 (RR) experiments (part 2 of 2)
# Batching, loss, prediction, training loop, submission. Run right after E0a.
# =====================================================================================

def identity_probe():
    """First 64 tokens of a CL training document, for the 'adapter = identity at init' check."""
    d = next(iter(train["CL"].values()))
    ids = [t for x in d["ids"] for t in x][:64]
    return torch.tensor([ids], device=device)


def count_params(model):
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad)
    tot = sum(p.numel() for p in model.parameters())
    print(f"trainable {tr/1e6:.3f}M / total {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")
    return tr, tot


def make_batches(examples, batch_size, shuffle):
    """[c-1] Batches of inputs of similar length, RIGHT-padded in to_tensors.
    Safe for c-1: a causal token never attends to the pads on its right, and pads are
    never inside a sentence span, so they reach neither the pooling nor the loss.
    (c-6 had no padding mask on a bidirectional encoder, so it batched only equal lengths.)"""
    exs = list(examples)
    if shuffle:
        random.shuffle(exs)
    out, pool = [], batch_size * 50
    for i in range(0, len(exs), pool):          # sort by length inside pools of 50 batches
        chunk = sorted(exs[i:i + pool], key=lambda ex: len(ex["ids"]))
        out += [chunk[j:j + batch_size] for j in range(0, len(chunk), batch_size)]
    if shuffle:
        random.shuffle(out)
    return out


def to_tensors(batch):
    T = max(len(ex["ids"]) for ex in batch)                             # [c-1] right-pad to the longest
    ids = torch.full((len(batch), T), PAD_ID, dtype=torch.long)
    for i, ex in enumerate(batch):
        ids[i, :len(ex["ids"])] = torch.tensor(ex["ids"])
    return {"ids": ids.to(device),
            "spans": [ex["spans"] for ex in batch],
            "labels": torch.tensor([l for ex in batch for l in ex["labels"]], device=device)}


def ce_loss(logits, labels, class_weights=None):
    """Plain sentence-level cross-entropy on gold labels."""
    return F.cross_entropy(logits.float(), labels, weight=class_weights, ignore_index=IGNORE)


@torch.no_grad()
def predict(model, examples, docs_by_domain, batch_size):
    model.eval()
    out = {dom: {k: [None] * len(d["text"]) for k, d in docs.items()} for dom, docs in docs_by_domain.items()}
    for batch in tqdm(make_batches(examples, batch_size, shuffle=False), desc="predict", leave=False):
        b = to_tensors(batch)
        with torch.autocast("cuda", dtype=torch.float16):
            pred = model(b["ids"], b["spans"]).argmax(-1).tolist()
        keys = [k for ex in batch for k in ex["keys"]]
        for (dom, doc_id, i), p in zip(keys, pred):
            out[dom][doc_id][i] = RR_LABELS[p]
    missing = sum(x is None for dom in out for v in out[dom].values() for x in v)
    assert missing == 0, f"{missing} sentences got no prediction"
    return out


# Training loop: keeps the epoch with the best dev macro-F1 over CL+IT together
def train_model(model, train_ex, dev_ex, dev_docs, loss_fn, *, lr, epochs, batch_size,
          weight_decay=0.0, grad_clip=1.0, lr_lambda=None, save_path="best_trainable.pt"):
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda) if lr_lambda else None
    scaler = torch.amp.GradScaler("cuda")
    torch.cuda.reset_peak_memory_stats()
    t0, best_f1, best_res, log, skipped = time.time(), -1.0, None, [], 0
    for epoch in range(epochs):
        model.train(); tot, nb = 0.0, 0
        pbar = tqdm(make_batches(train_ex, batch_size, shuffle=True), desc=f"epoch {epoch + 1}/{epochs}")
        for batch in pbar:
            b = to_tensors(batch)
            with torch.autocast("cuda", dtype=torch.float16):
                logits = model(b["ids"], b["spans"])
            loss = loss_fn(logits.float(), b)
            opt.zero_grad(set_to_none=True)
            if not torch.isfinite(loss):
                skipped += 1
                continue
            scaler.scale(loss).backward()
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(params, grad_clip)
            scaler.step(opt); scaler.update()
            if sched: sched.step()
            tot += loss.item(); nb += 1
            pbar.set_postfix(loss=f"{tot / nb:.4f}", skipped=skipped)
        res = evaluate(predict(model, dev_ex, dev_docs, batch_size), dev_docs, show=False)
        log.append({"epoch": epoch + 1, "train_loss": tot / max(1, nb), "dev_f1_ALL": res["ALL"]["f1"],
                    "dev_f1_CL": res["CL"]["f1"], "dev_f1_IT": res["IT"]["f1"], "skipped": skipped})
        print(log[-1])
        if res["ALL"]["f1"] > best_f1:
            best_f1, best_res = res["ALL"]["f1"], res
            torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, save_path)
    print(f"best dev macro-F1 ALL {best_f1:.4f} (CL {best_res['CL']['f1']:.4f}, IT {best_res['IT']['f1']:.4f}) | "
          f"train time {(time.time() - t0) / 60:.1f} min | peak GPU mem {torch.cuda.max_memory_allocated() / 2**30:.2f} GB")
    per_label_table(best_res)
    return log, best_res


@torch.no_grad()
def time_inference(model, examples, docs_by_domain, batch_size):
    torch.cuda.synchronize(); t = time.time()
    predict(model, examples, docs_by_domain, batch_size)
    torch.cuda.synchronize(); return time.time() - t


def write_submission(model, build_kwargs, batch_size, best_path=None, out_dir="/kaggle/working"):
    """Reload the best trainable weights, predict CL+IT test, write prediction.json + submission.zip."""
    if best_path:
        model.load_state_dict(torch.load(best_path), strict=False)
    test_ex = build_examples(test, **build_kwargs)
    write_prediction(predict(model, test_ex, test, batch_size), test, out_dir=out_dir)

In [11]:
# =====================================================================================
# CELL E2 — LORMA on the c-1 decoder for Rhetorical Role prediction
# Run after Cells 0, A, B, C, D, E0a and E0b. To run it again from scratch, re-run Cell A, E0a
# and E0b first (LORMA wraps the decoder's linears in place).
# Head and hyperparameters: Rudra's cell, unchanged. Lines marked [c-1] differ from c-6.
# =====================================================================================

CONTEXT_MODE    = "document"   # <-- PICK: "sentence" (each sentence alone) or "document" (neighbouring sentences in the same input)
MAX_SENT_TOKENS = 200    # sentence mode: keep the first N tokens of a sentence (Phase 2 WINDOW)
WINDOW          = 200    # document mode: tokens per window (Phase 2)
STRIDE          = 10     # document mode: step between window starts (Phase 2)
TRAIN_DOMAINS   = ("CL", "IT")   # <-- PICK: ("CL", "IT") to train on both, or ("CL",) / ("IT",) for one domain
LORMA_RANK      = 80     # Phase 2
LORMA_TARGETS   = ("q_proj", "k_proj", "v_proj", "out_proj")   # [c-1] c-1 attention linears: separate q/k/v (c-6 had one fused qkv_proj)
LR              = 1e-4   # Phase 2
EPOCHS          = 50     # Phase 2
BATCH_SIZE      = 64     # Phase 2 (sentence mode: sentences per step; document mode: windows per step)
WEIGHT_DECAY    = 0.0    # Phase 2
LR_LAMBDA       = None



# -------------------------------------------------------------------------------------
# Class weights from the training label counts (same block as the c-6 thread)
# -------------------------------------------------------------------------------------
WEIGHT_SCHEME = "sqrt_inverse"   # <-- PICK: "inverse" (1/count), "sqrt_inverse" (1/sqrt(count)) or "none"
assert WEIGHT_SCHEME in ("inverse", "sqrt_inverse", "none"), "set WEIGHT_SCHEME"
_cnt = Counter(l for d in TRAIN_DOMAINS for doc in train[d].values() for l in doc["labels"])
_counts = torch.tensor([_cnt.get(l, 0) for l in RR_LABELS], dtype=torch.float)
_present = _counts > 0                       # labels that never occur in train get weight 0
if WEIGHT_SCHEME == "none":
    CLASS_WEIGHTS = None
else:
    _w = torch.zeros(len(RR_LABELS))
    _w[_present] = 1.0 / _counts[_present] if WEIGHT_SCHEME == "inverse" else 1.0 / _counts[_present].sqrt()
    _w[_present] = _w[_present] / _w[_present].mean()   # average weight 1, so the loss scale stays similar
    CLASS_WEIGHTS = _w.to(device)
    for l, n, w in zip(RR_LABELS, _counts.tolist(), _w.tolist()):
        print(f"  {l:24s} train n={int(n):6d}  weight={w:.3f}")





SEED            = 42
for _n in ["CONTEXT_MODE", "TRAIN_DOMAINS", "LORMA_RANK", "LORMA_TARGETS", "LR", "EPOCHS", "BATCH_SIZE"]:
    assert globals()[_n] is not None, f"set {_n}"
if CONTEXT_MODE == "sentence":
    assert MAX_SENT_TOKENS is not None and MAX_SENT_TOKENS <= cfg.max_seq_len, "set MAX_SENT_TOKENS"
    BUILD = dict(mode="sentence", max_sent_tokens=MAX_SENT_TOKENS)
else:
    assert CONTEXT_MODE == "document" and WINDOW is not None and STRIDE is not None
    assert STRIDE < WINDOW <= cfg.max_seq_len
    BUILD = dict(mode="document", window=WINDOW, stride=STRIDE)
random.seed(SEED); torch.manual_seed(SEED)
RUN_NAME = f"c1_lorma_mult_{CONTEXT_MODE}"   # separate files from the LoRA run (c1_lorma_...)


# [L] LORMA layer (identity variant) — Rudra's code from 15 Oct, pasted as sent.
#   Still to do (Rudra): the two lines marked <-- EDIT. Until then forward() ignores W,
#   so the identity check below stops the cell before training.

# [L] LORMA layer (identity variant)
class LORMALinear(nn.Module):
    def __init__(self, base: nn.Linear, rank: int):
        super().__init__()
        self.base = base

        # Freeze pretrained weights
        for p in self.base.parameters():
            p.requires_grad = False

        self.in_features = base.in_features
        self.out_features = base.out_features
        self.rank = rank

        # A: (rank, out_features), since A operates on Wx
        # B: (out_features, rank)
        self.lorma_A = nn.Parameter(
            torch.empty(self.rank, self.out_features)
        )
        self.lorma_B = nn.Parameter(
            torch.empty(self.out_features, self.rank)
        )

        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        nn.init.zeros_(self.lorma_B)

    def forward(self, x):
        # Compute pretrained linear output without bias: Wx
        wx = F.linear(x, self.base.weight, bias=None)

        # Low-rank transformation of Wx: BA(Wx)
        lorma_out = F.linear(
            F.linear(wx, self.lorma_A),
            self.lorma_B
        )

        # Return Wx + BA(Wx) + b
        if self.base.bias is not None:
            return wx + lorma_out + self.base.bias

        return wx + lorma_out



# Freeze c-1, wrap the chosen linears, check identity at init
lm.eval()
probe = identity_probe()
with torch.no_grad():
    ref = features(probe).float()          # [c-1] Cell B features() = decoder hidden states

for p in lm.parameters():
    p.requires_grad = False
n_wrapped = wrap_linears(lm, LORMA_TARGETS, lambda lin: LORMALinear(lin, LORMA_RANK).to(device))

with torch.no_grad():
    diff = (features(probe).float() - ref).abs().max().item()
print(f"wrapped {n_wrapped} linears | LORMA identity-at-init max |diff| = {diff:.2e} (should be ~0)")
assert diff < 1e-3, "LORMA layer is not the identity at init: finish the two <-- EDIT lines in LORMALinear"


# [P] Sentence representation: h is (L, 1024), the tokens of ONE sentence   [c-1] 1024, not 560
def pool_sentence(h):
    return h.mean(0)


# [N] Head — Rudra's Phase 2 head (Dropout 0.1 + Linear), input 1024 for c-1, 13 outputs
head = nn.Sequential(
    nn.Dropout(0.1),
    nn.Linear(FEAT_DIM, N_LABELS)
)
model = RRModel(lm, pool_sentence, head).to(device)
count_params(model)


# Data, training, scoring (plumbing from E0). Dev is always scored on BOTH domains.
train_ex = build_examples({d: train[d] for d in TRAIN_DOMAINS}, **BUILD)
dev_ex = build_examples(dev, **BUILD)
print(f"train inputs {len(train_ex)} | dev inputs {len(dev_ex)}")

log, best_res = train_model(
    model, train_ex, dev_ex, dev,
    loss_fn=lambda logits, b: ce_loss(logits, b["labels"], CLASS_WEIGHTS),
    lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, weight_decay=WEIGHT_DECAY,
    lr_lambda=LR_LAMBDA, save_path=f"/kaggle/working/{RUN_NAME}_best_trainable.pt")

# Evidence for the report: confusion matrices with the best weights, inference time
model.load_state_dict(torch.load(f"/kaggle/working/{RUN_NAME}_best_trainable.pt"), strict=False)
dev_pred = predict(model, dev_ex, dev, BATCH_SIZE)
for dom in DOMAINS:
    confusion(dev_pred, dev, dom)
print(f"dev inference time {time_inference(model, dev_ex, dev, BATCH_SIZE):.1f} s")

# Predict CL + IT test with the best weights and package
write_submission(model, BUILD, BATCH_SIZE, best_path=f"/kaggle/working/{RUN_NAME}_best_trainable.pt")

  Fact                     train n=  4098  weight=0.192
  Issue                    train n=   107  weight=1.186
  ArgumentPetitioner       train n=  1613  weight=0.305
  ArgumentRespondent       train n=  1348  weight=0.334
  Statute                  train n=  1099  weight=0.370
  Dissent                  train n=   284  weight=0.728
  PrecedentReliedUpon      train n=  1945  weight=0.278
  PrecedentNotReliedUpon   train n=   569  weight=0.514
  PrecedentOverruled       train n=     3  weight=7.080
  RulingByLowerCourt       train n=   874  weight=0.415
  RatioOfTheDecision       train n=  4301  weight=0.187
  RulingByPresentCourt     train n=   307  weight=0.700
  None                     train n=   297  weight=0.712
wrapped 32 linears | LORMA identity-at-init max |diff| = 0.00e+00 (should be ~0)
trainable 5.256M / total 116.24M (4.52%)


CL examples (document):   0%|          | 0/40 [00:00<?, ?it/s]

IT examples (document):   0%|          | 0/40 [00:00<?, ?it/s]

CL examples (document):   0%|          | 0/5 [00:00<?, ?it/s]

IT examples (document):   0%|          | 0/5 [00:00<?, ?it/s]

train inputs 16494 | dev inputs 2093


... (154 outputs left out)

CL examples (document):   0%|          | 0/5 [00:00<?, ?it/s]

IT examples (document):   0%|          | 0/5 [00:00<?, ?it/s]

predict:   0%|          | 0/34 [00:00<?, ?it/s]

10 docs, 2197 sentences -> /kaggle/working/submission.zip
